<a href="https://colab.research.google.com/github/Ravindra1972/Anaytics-in-finance-using-Python/blob/main/Bond%20Pricing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install QuantLib-Python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 67.2 MB/s eta 0:00:00


In [14]:
import QuantLib as ql

# Valuation date
today = ql.Date(23, 6, 2026)
ql.Settings.instance().evaluationDate = today

# Bond details
issue_date = ql.Date(23, 6, 2023)
maturity_date = ql.Date(23, 6, 2031)
face_amount = 100.0
coupon_rate = 0.06 # Corrected typo from copon_rate
calendar = ql.TARGET()
day_count = ql.Thirty360(ql.Thirty360.BondBasis) # Corrected day count convention
business_convention = ql.Unadjusted
frequency = ql.Semiannual

# Flat discount curve
flat_rate = 0.05
yield_curve = ql.FlatForward(today, flat_rate, ql.Actual365Fixed(), ql.Compounded, ql.Annual)
discount_curve = ql.YieldTermStructureHandle(yield_curve)

# Bond schedule
schedule = ql.Schedule(
    issue_date,
    maturity_date,
    ql.Period(frequency),
    calendar,
    business_convention,
    business_convention,
    ql.DateGeneration.Backward,
    False
)

# Fixed-rate bond
bond = ql.FixedRateBond(
    2,                  # settlement days
    face_amount,
    schedule,
    [coupon_rate],
    day_count
)

# Pricing engine
engine = ql.DiscountingBondEngine(discount_curve)
bond.setPricingEngine(engine)

# Results
clean_price = bond.cleanPrice()
dirty_price = bond.dirtyPrice()
yield_rate = bond.bondYield(day_count, ql.Compounded, frequency)

print("Clean price:", clean_price)
print("Dirty price:", dirty_price)
print("Yield:", yield_rate)

Clean price: 104.6283594553022
Dirty price: 104.66169278863553
Yield: 0.04942145347595215


In [18]:
import QuantLib as ql

# Evaluation date
today = ql.Date(23, 6, 2026)
ql.Settings.instance().evaluationDate = today

calendar = ql.TARGET()
settlement_days = 2 # Corrected typo from sستtlement_days
face_amount = 100.0

# Flat discount curve
flat_rate = 0.05
discount_curve = ql.YieldTermStructureHandle(
    ql.FlatForward(today, flat_rate, ql.Actual365Fixed())
)

# Floating index setup
index_curve = ql.YieldTermStructureHandle(
    ql.FlatForward(today, 0.04, ql.Actual365Fixed())
)

ibor_index = ql.Euribor3M(index_curve)

# Bond schedule
issue_date = ql.Date(23, 6, 2024)
maturity_date = ql.Date(23, 6, 2030)

schedule = ql.Schedule(
    issue_date,
    maturity_date,
    ql.Period(ql.Semiannual),
    calendar,
    ql.ModifiedFollowing,
    ql.ModifiedFollowing,
    ql.DateGeneration.Backward,
    False
)

# Floating-rate bond: index + spread
spread = 0.0025  # 25 bps
bond = ql.FloatingRateBond(
    settlement_days,
    face_amount,
    schedule,
    ibor_index,
    ql.Actual360(),
    ql.ModifiedFollowing,
    fixingDays=2,
    spreads=[spread]
)

# QuantLib requires explicit fixings for past dates relative to the evaluation date.
# The previous RuntimeError indicated a missing fixing for June 19th, 2026.
# Add the specific fixing for the required date, using the rate from the index_curve.
ibor_index.addFixing(ql.Date(19, 6, 2026), 0.04)

# Pricing engine
engine = ql.DiscountingBondEngine(discount_curve)
bond.setPricingEngine(engine)

print("Clean price:", bond.cleanPrice())
print("Dirty price:", bond.dirtyPrice())
print("Accrued:", bond.accruedAmount())

Clean price: 97.25274051294767
Dirty price: 97.27635162405878
Accrued: 0.02361111111111111


In [19]:
import QuantLib as ql

# Valuation date
today = ql.Date(23, 6, 2026)
ql.Settings.instance().evaluationDate = today

calendar = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
settlement_days = 2
face_amount = 100.0

# Discount curve
discount_curve = ql.YieldTermStructureHandle(
    ql.FlatForward(today, 0.05, ql.Actual365Fixed())
)

# SOFR curve for forwards / coupon projection
sofr_curve = ql.YieldTermStructureHandle(
    ql.FlatForward(today, 0.045, ql.Actual365Fixed())
)

# SOFR index
sofr = ql.Sofr(sofr_curve)

# Bond schedule
issue_date = ql.Date(23, 6, 2024)
maturity_date = ql.Date(23, 6, 2030)

schedule = ql.Schedule(
    issue_date,
    maturity_date,
    ql.Period(ql.Quarterly),
    calendar,
    ql.ModifiedFollowing,
    ql.ModifiedFollowing,
    ql.DateGeneration.Backward,
    False
)

# Floating-rate bond: SOFR + spread
spread = 0.005  # 50 bps
bond = ql.FloatingRateBond(
    settlement_days,
    face_amount,
    schedule,
    sofr,
    ql.Actual360(),
    ql.ModifiedFollowing,
    fixingDays=0,
    spreads=[spread]
)

# Add historical fixings if any coupon has already started
for d in schedule:
    if d < today:
        try:
            sofr.addFixing(d, 0.045)
        except:
            pass

bond.setPricingEngine(ql.DiscountingBondEngine(discount_curve))

print("Clean price:", bond.cleanPrice())
print("Dirty price:", bond.dirtyPrice())
print("Accrued:", bond.accruedAmount())

Clean price: 100.00329380651921
Dirty price: 100.03086948736997
Accrued: 0.02757568085075802


In [21]:
import QuantLib as ql

# Evaluation date
today = ql.Date(23, 6, 2026)
ql.Settings.instance().evaluationDate = today

calendar = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
settlement_days = 2
day_count = ql.Actual360()

# Market quotes
deposits = [
    (ql.Period("1M"), 0.0500),
    (ql.Period("3M"), 0.0515),
    (ql.Period("6M"), 0.0530),
]

swaps = [
    (ql.Period("1Y"), 0.0540),
    (ql.Period("2Y"), 0.0560),
    (ql.Period("3Y"), 0.0575),
    (ql.Period("5Y"), 0.0590),
]

helpers = []

# Deposit helpers
for tenor, rate in deposits:
    quote = ql.QuoteHandle(ql.SimpleQuote(rate))
    helper = ql.DepositRateHelper(
        quote,
        tenor,
        settlement_days,
        calendar,
        ql.ModifiedFollowing,
        False,
        day_count
    )
    helpers.append(helper)

# Swap helpers
fixed_leg_frequency = ql.Annual
fixed_leg_convention = ql.Unadjusted
fixed_leg_day_count = ql.Thirty360(ql.Thirty360.BondBasis) # Corrected: Added convention argument
float_index = ql.USDLibor(ql.Period("3M"))

for tenor, rate in swaps:
    quote = ql.QuoteHandle(ql.SimpleQuote(rate))
    helper = ql.SwapRateHelper(
        quote,
        tenor,
        calendar,
        fixed_leg_frequency,
        fixed_leg_convention,
        fixed_leg_day_count,
        float_index
    )
    helpers.append(helper)

# Bootstrap curve
curve = ql.PiecewiseLinearZero(today, helpers, ql.Actual365Fixed())
curve.enableExtrapolation()
curve_handle = ql.YieldTermStructureHandle(curve)

# Example: inspect zero rates
for d in [ql.Date(23, 9, 2026), ql.Date(23, 6, 2027), ql.Date(23, 6, 2028)]:
    zr = curve.zeroRate(d, ql.Actual365Fixed(), ql.Continuous).rate()
    print(d, zr)

September 23rd, 2026 0.05180498907018257
June 23rd, 2027 0.05259051733412494
June 23rd, 2028 0.05436714267558694
